# Chapter 04 — Flash Attention

**Goal:** Understand why standard attention doesn't scale, how Flash
Attention solves the O(S^2) memory problem, and quantify the speedups.

**Reference model:** LLaMA-3.2-1B  
d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers

**Hardware:** A100-80GB — 1,774 GB/s bandwidth, 312 TFLOPS FP16 (237 TFLOPS effective), ~134 FLOPS/byte ridge point

## 1 — The Problem: O(S²) Memory

Standard attention materializes the full S × S score matrix in HBM:

```
scores = Q @ K^T           → shape (n_heads, S, S)
probs  = softmax(scores)   → shape (n_heads, S, S)  (another S×S matrix!)
output = probs @ V         → shape (n_heads, S, d_head)
```

Both `scores` and `probs` are S × S matrices stored in HBM.
For 32 heads in FP16, that's `32 × S × S × 2 bytes` — **per matrix**.

In [ ]:
# ── O(S^2) Memory Calculator ─────────────────────────────────────────

n_heads     = 32
dtype_bytes = 2      # FP16

seq_lengths = [512, 2048, 8192, 32768, 131072]

def attention_matrix_bytes(n_heads, seq_len, dtype_bytes):
    # YOUR CODE
    # Hint: n_heads * seq_len * seq_len * dtype_bytes
    # We need TWO such matrices (scores + probs), so multiply by 2
    # Expected: S=512 → 32MB, S=2048 → 512MB, S=8192 → 8GB, S=32K → 128GB
    pass

print("Score + Prob matrices in HBM (32 heads, FP16):")
print(f"{'seq_len':>8s}  {'Memory':>10s}  {'Fits A100?':>10s}")
print("-" * 35)
for s in seq_lengths:
    mem = attention_matrix_bytes(n_heads, s, dtype_bytes)
    mem_gb = mem / 1e9
    fits = "Yes" if mem_gb < 80 else "NO"
    if mem_gb < 1:
        print(f"{s:>8d}  {mem/1e6:>8.0f} MB  {fits:>10s}")
    else:
        print(f"{s:>8d}  {mem_gb:>8.1f} GB  {fits:>10s}")

print("\nO(S^2) memory simply does not scale to long contexts.")

## 2 — Standard Attention Step by Step

Let's implement standard attention manually and analyze the
arithmetic intensity of each step.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

def standard_attention(Q, K, V):
    """Standard scaled dot-product attention. Shape: (batch, heads, seq, d_head)"""
    d_head = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)  # (B, H, S, S)
    probs = torch.softmax(scores, dim=-1)                # (B, H, S, S)
    output = probs @ V                                    # (B, H, S, d)
    return output

# Test
B, H, S, D = 1, 32, 512, 64
Q = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16)
K = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16)
V = torch.randn(B, H, S, D, device='cuda', dtype=torch.float16)

out = standard_attention(Q, K, V)
print(f"Output shape: {out.shape}")

# ── Arithmetic Intensity per step ────────────────────────────────────
# YOUR CODE: compute AI for each step
# Step 1: Q @ K^T
#   FLOPs = H * S * S * D * 2  (matmul)
#   Bytes = read Q (H*S*D*2) + read K (H*S*D*2) + write scores (H*S*S*2)
#   AI = FLOPs / Bytes
#
# Step 2: softmax(scores)
#   FLOPs ≈ H * S * S * 5  (max, sub, exp, sum, div — ~5 ops per element)
#   Bytes = read scores (H*S*S*2) + write probs (H*S*S*2)
#   AI = FLOPs / Bytes
#
# Step 3: probs @ V
#   FLOPs = H * S * D * S * 2  (matmul)
#   Bytes = read probs (H*S*S*2) + read V (H*S*D*2) + write output (H*S*D*2)
#   AI = FLOPs / Bytes

ridge_point = 134  # A100
print(f"\nA100 ridge point: {ridge_point} FLOP/byte")
print("Hint: softmax is extremely memory-bound (AI ≈ 2.5 FLOP/byte)")
print("Hint: the matmuls are also memory-bound at typical seq lengths")

## 3 — Why Softmax Blocks Tiling

### The core obstacle

We'd love to tile the computation — process blocks of K/V at a time,
keep Q in SRAM, never write the S×S matrix to HBM.

**Problem: softmax needs global information.**

```python
softmax(x_i) = exp(x_i) / sum(exp(x_j) for all j)
```

To compute the output for *any* position, we need the **sum over ALL
positions** in the denominator. This forces us to:

1. Compute ALL scores first (the full S×S matrix)
2. Find the global max (for numerical stability)
3. Compute exp and sum
4. Normalize

Each step reads/writes the full S×S matrix from/to HBM.

**This is why naive tiling doesn't work** — you can't normalize a
partial softmax without knowing what the rest of the row looks like.

...unless you use **online softmax**.

## 4 — Online Softmax (Milakov & Gimelshein 2018)

### The key insight: maintain running statistics

Instead of needing all values upfront, process them one (or one block)
at a time, maintaining:

- `m` — running max so far
- `l` — running sum of exp(x_i - m) so far

When a new block arrives with a larger max, we *correct* the running
sum:

```python
m_new = max(m_old, max(new_block))
l_new = l_old * exp(m_old - m_new) + sum(exp(new_block - m_new))
```

This lets us process attention scores in blocks without ever
materializing the full S×S matrix.

In [ ]:
import torch

def naive_softmax(x):
    """Standard 3-pass softmax: max, exp-sum, normalize."""
    m = x.max(dim=-1, keepdim=True).values
    e = torch.exp(x - m)
    return e / e.sum(dim=-1, keepdim=True)

def online_softmax(x, block_size=64):
    """Online softmax: process in blocks, maintain running max and sum."""
    S = x.shape[-1]
    # YOUR CODE
    # Hint:
    # 1. Initialize m = -inf, l = 0, output = zeros
    # 2. For each block of size block_size:
    #    a. m_new = max(m, block.max())
    #    b. correction = exp(m - m_new)   # rescale old statistics
    #    c. l = l * correction + sum(exp(block - m_new))
    #    d. m = m_new
    # 3. After all blocks: probs_i = exp(x_i - m) / l
    pass

# Verify correctness
x = torch.randn(1, 256)
naive_out = naive_softmax(x)
online_out = online_softmax(x, block_size=64)
print(f"Max absolute difference: {(naive_out - online_out).abs().max().item():.2e}")
print("Should be < 1e-5 (FP32) — online softmax is numerically equivalent.")

## 5 — Flash Attention Algorithm

Flash Attention (Dao et al. 2022) combines online softmax with tiling:

1. **Tile Q** into blocks of rows (B_r rows at a time)
2. For each Q block, iterate over **K/V blocks** (B_c columns at a time):
   - Load Q block, K block, V block into SRAM
   - Compute partial scores: Q_block @ K_block^T
   - Update running softmax statistics (online softmax)
   - Accumulate: output += softmax_weights @ V_block
   - **Never write the S×S matrix to HBM**
3. Write final output block to HBM

Memory: O(S) instead of O(S²) — only the output and softmax stats.
FLOPs: Same as standard attention — no free lunch on compute.
Speed: Faster because we avoid HBM reads/writes of the S×S matrix.

In [ ]:
import torch
import math

def flash_attention_educational(Q, K, V, block_size=64):
    """
    Simplified Flash Attention in Python (educational, NOT for performance).
    Shape: (seq_len, d_head) — single head, no batch.
    """
    S, D = Q.shape
    scale = 1.0 / math.sqrt(D)

    # Output accumulator and softmax statistics
    O = torch.zeros(S, D, dtype=Q.dtype, device=Q.device)
    m = torch.full((S, 1), float('-inf'), dtype=Q.dtype, device=Q.device)  # running max
    l = torch.zeros(S, 1, dtype=Q.dtype, device=Q.device)                 # running sum

    # YOUR CODE
    # Hint: iterate over K/V in blocks
    # for j in range(0, S, block_size):
    #     K_block = K[j:j+block_size]        # (B_c, D)
    #     V_block = V[j:j+block_size]        # (B_c, D)
    #     scores = Q @ K_block.T * scale     # (S, B_c)
    #
    #     # Online softmax update
    #     m_new = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)
    #     correction = torch.exp(m - m_new)
    #     p = torch.exp(scores - m_new)      # unnormalized weights for this block
    #
    #     # Rescale old output and accumulate
    #     l_new = l * correction + p.sum(dim=-1, keepdim=True)
    #     O = O * (l * correction / l_new) + (p / l_new) @ V_block
    #
    #     m = m_new
    #     l = l_new
    # return O
    pass

# Verify against standard attention
S, D = 256, 64
Q = torch.randn(S, D, device='cuda', dtype=torch.float32)
K = torch.randn(S, D, device='cuda', dtype=torch.float32)
V = torch.randn(S, D, device='cuda', dtype=torch.float32)

# Standard attention
scores = Q @ K.T / math.sqrt(D)
probs = torch.softmax(scores, dim=-1)
ref_output = probs @ V

# Flash attention (educational)
flash_output = flash_attention_educational(Q, K, V, block_size=64)

if flash_output is not None:
    diff = (ref_output - flash_output).abs().max().item()
    print(f"Max absolute difference: {diff:.2e}")
    print("Should be < 1e-5 — same result, different algorithm.")
else:
    print("Implement flash_attention_educational above!")

## 6 — Benchmark: Flash vs Standard

PyTorch 2.0+ includes `torch.nn.functional.scaled_dot_product_attention`
(SDPA) which automatically dispatches to Flash Attention when possible.

Let's benchmark it against the naive implementation.

In [ ]:
import torch
import torch.nn.functional as F
import time

def benchmark_attention(seq_len, n_heads=32, d_head=64, warmup=5, trials=20):
    """Benchmark standard vs Flash Attention."""
    B = 1
    Q = torch.randn(B, n_heads, seq_len, d_head, device='cuda', dtype=torch.float16)
    K = torch.randn(B, n_heads, seq_len, d_head, device='cuda', dtype=torch.float16)
    V = torch.randn(B, n_heads, seq_len, d_head, device='cuda', dtype=torch.float16)

    # ── Standard attention (naive) ──
    def naive_attn():
        s = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
        p = torch.softmax(s, dim=-1)
        return p @ V

    # ── Flash Attention via SDPA ──
    def flash_attn():
        return F.scaled_dot_product_attention(Q, K, V)

    results = {}
    for name, fn in [("naive", naive_attn), ("flash", flash_attn)]:
        # Warmup
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()

        # YOUR CODE: benchmark with torch.cuda.Event for accurate timing
        # Hint:
        # start = torch.cuda.Event(enable_timing=True)
        # end = torch.cuda.Event(enable_timing=True)
        # start.record()
        # for _ in range(trials):
        #     fn()
        # end.record()
        # torch.cuda.synchronize()
        # ms = start.elapsed_time(end) / trials
        # results[name] = ms
        pass

    # Also measure peak memory
    for name, fn in [("naive", naive_attn), ("flash", flash_attn)]:
        torch.cuda.reset_peak_memory_stats()
        fn()
        torch.cuda.synchronize()
        results[f"{name}_mem_mb"] = torch.cuda.max_memory_allocated() / 1e6

    return results

# Run benchmarks
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]
print(f"{'seq_len':>8s}  {'naive (ms)':>10s}  {'flash (ms)':>10s}  {'speedup':>8s}  {'naive MB':>10s}  {'flash MB':>10s}  {'mem save':>8s}")
print("-" * 80)

for s in seq_lengths:
    try:
        r = benchmark_attention(s)
        if r.get('naive') and r.get('flash'):
            speedup = r['naive'] / r['flash']
            mem_save = r['naive_mem_mb'] / r['flash_mem_mb']
            print(f"{s:>8d}  {r['naive']:>10.2f}  {r['flash']:>10.2f}  {speedup:>7.1f}×  {r['naive_mem_mb']:>10.0f}  {r['flash_mem_mb']:>10.0f}  {mem_save:>7.1f}×")
    except torch.cuda.OutOfMemoryError:
        print(f"{s:>8d}  OOM")
        torch.cuda.empty_cache()

## 7 — Roofline Analysis

Flash Attention does the **same FLOPs** as standard attention.
The win is fewer **bytes transferred** to/from HBM.

On the roofline, this means Flash Attention moves the operation
**to the RIGHT** (higher arithmetic intensity) — same numerator,
smaller denominator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A100 parameters
bw_gb_s = 1774       # GB/s
peak_tflops = 237    # effective TFLOPS FP16
ridge_point = peak_tflops * 1e3 / bw_gb_s  # ~134 FLOP/byte

# ── Attention: Standard vs Flash ─────────────────────────────────────
H, D = 32, 64
seq_lengths = [512, 1024, 2048, 4096, 8192]

# YOUR CODE: Calculate AI for standard and Flash attention
# Standard attention:
#   FLOPs = H * (2*S*S*D + 5*S*S + 2*S*S*D)  ≈ H * 4*S*S*D (dominated by matmuls)
#   Bytes = read Q,K,V (3*H*S*D*2) + write/read scores (2*H*S*S*2)
#           + write/read probs (2*H*S*S*2) + write output (H*S*D*2)
#         ≈ 8*H*S*D + 8*H*S*S  (in bytes, with FP16)
#   AI_standard = FLOPs / Bytes
#
# Flash Attention:
#   FLOPs = same as standard (no free lunch on compute)
#   Bytes = read Q,K,V (3*H*S*D*2) + write output (H*S*D*2)
#         = 8*H*S*D  (NO S×S matrices in HBM!)
#   AI_flash = FLOPs / Bytes

# Plot roofline with both points
ai_range = np.logspace(-1, 4, 500)
perf = np.minimum(ai_range * bw_gb_s / 1e3, peak_tflops)

fig, ax = plt.subplots(figsize=(12, 7))
ax.loglog(ai_range, perf, 'k-', linewidth=2, label='A100 Roofline')
ax.axvline(ridge_point, color='gray', linestyle='--', alpha=0.5, label=f'Ridge: {ridge_point:.0f} FLOP/byte')

# Plot standard and flash attention points for each seq length
# (Fill in after computing AI values above)
# for i, s in enumerate(seq_lengths):
#     ax.plot(ai_std[i], perf_std[i], 'rv', markersize=10)
#     ax.plot(ai_flash[i], perf_flash[i], 'g^', markersize=10)
#     ax.annotate(f'S={s}', ...)

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title('Standard vs Flash Attention on A100 Roofline', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 8 — Flash Attention 2 and 3

### Flash Attention 2 (Dao 2023)

Key improvements over FA1:
- **Better work partitioning** — parallelize over sequence length, not batch×heads
- **Reduced non-matmul FLOPs** — fewer rescaling operations in the online softmax
- **Better GPU occupancy** — more warps doing useful work
- Result: ~2× faster than FA1, reaching 50-73% of peak TFLOPS on A100

### Flash Attention 3 (Dao et al. 2024)

Designed for NVIDIA Hopper (H100):
- **Asynchronous softmax** — overlap softmax with GEMM using Hopper's
  asynchronous execution (WGMMA + TMA)
- **FP8 support** — use low-precision where possible, with block
  quantization to maintain accuracy
- **Incoherent processing** — apply random orthogonal matrix to smooth
  outlier distributions before FP8 quantization
- Result: up to 740 TFLOPS on H100 (75% utilization)

### The progression:

| Version | A100 TFLOPS | % of peak | Key innovation |
|---------|-------------|-----------|----------------|
| Standard | ~50 | ~16% | Baseline |
| FA1 | ~120 | ~38% | Tiling + online softmax |
| FA2 | ~180 | ~58% | Better parallelism |
| FA3 (H100) | ~740 | ~75% | Async + FP8 |

## 9 — When Flash Attention Helps Most

Flash Attention's advantage grows with sequence length because the
S×S memory it eliminates grows quadratically, while the actual
computation grows the same way.

At short sequences, the overhead of tiling may not pay off.
At long sequences, it's the difference between OOM and running.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Theoretical speedup from eliminating HBM traffic ─────────────────
H, D = 32, 64
dtype = 2  # FP16 bytes

seq_lengths = np.array([128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768])

def hbm_bytes_standard(S, H, D, dtype):
    """Total HBM bytes for standard attention."""
    # YOUR CODE
    # Hint: read Q,K,V + write scores + read scores + write probs + read probs + write output
    # = 3*H*S*D*dtype (Q,K,V) + 4*H*S*S*dtype (scores+probs, write+read each) + H*S*D*dtype (output)
    pass

def hbm_bytes_flash(S, H, D, dtype):
    """Total HBM bytes for Flash Attention (no S×S in HBM)."""
    # YOUR CODE
    # Hint: read Q,K,V + write output = 3*H*S*D*dtype + H*S*D*dtype = 4*H*S*D*dtype
    pass

speedups = []
for s in seq_lengths:
    std_bytes = hbm_bytes_standard(s, H, D, dtype)
    flash_bytes = hbm_bytes_flash(s, H, D, dtype)
    if std_bytes and flash_bytes:
        speedups.append(std_bytes / flash_bytes)
    else:
        speedups.append(1.0)

plt.figure(figsize=(10, 5))
plt.plot(seq_lengths, speedups, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Sequence Length', fontsize=12)
plt.ylabel('Theoretical HBM Bandwidth Reduction', fontsize=12)
plt.title('Flash Attention Advantage vs Sequence Length', fontsize=14)
plt.xscale('log', base=2)
plt.grid(True, alpha=0.3)
for s, sp in zip(seq_lengths, speedups):
    plt.annotate(f'{sp:.1f}×', (s, sp), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("Longer sequences → bigger wins. At S=32K, Flash eliminates ~128× HBM traffic.")

## Revision Notes

| Topic | Key Insight | Formula / Number |
|-------|-------------|------------------|
|       |             |                  |
|       |             |                  |
|       |             |                  |
|       |             |                  |
|       |             |                  |